In [1]:
# Update transformers to latest version that supports Gemma 4
!pip install -q --upgrade transformers
!pip install -q accelerate gradio

print("✅ Libraries updated!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 76.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 6.4 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 62.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 1.7 MB/s eta 0:00:00ta 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is 

In [2]:
import os
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# ✅ Find the model path automatically
def find_model_path():
    base = "/kaggle/input"
    for folder in os.listdir(base):
        folder_path = os.path.join(base, folder)
        print(f"Found input folder: {folder_path}")
    
    # Common paths Kaggle uses
    possible_paths = [
        "/kaggle/input/gemma-4/transformers/gemma-4-e4b-it/1",
        "/kaggle/input/gemma4/transformers/gemma-4-e4b-it/1",
        "/kaggle/input/gemma-4/transformers/e4b-it/1",
        "/kaggle/input/gemma4/transformers/e4b-it/1",
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"✅ Model found at: {path}")
            return path
    
    print("❌ Model not found in above paths.")
    print("📂 Searching deeper...")
    for root, dirs, files in os.walk("/kaggle/input"):
        level = root.replace("/kaggle/input", "").count(os.sep)
        if level < 4:
            print(f"   {root}")
    return None

model_path = find_model_path()

Found input folder: /kaggle/input/models
❌ Model not found in above paths.
📂 Searching deeper...
   /kaggle/input
   /kaggle/input/models
   /kaggle/input/models/google
   /kaggle/input/models/google/gemma-4


In [3]:
import os
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# ✅ Correct path found from your output
model_path = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"

print(f"Loading model from: {model_path}")

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
print("✅ Model loaded and ready!")

Loading model from: /kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ Model loaded and ready!


In [4]:
import gradio as gr

SYSTEM_PROMPT = """
You are a helpful, simple, and safe medical assistant for rural Pakistan.
Your job is to help patients and community health workers.

RULES you must follow:
1. Always respond in BOTH Urdu and English
2. Keep language very simple — patients may not be educated
3. NEVER diagnose — only suggest possibilities
4. Always say "please see a doctor" for serious symptoms
5. Be kind, caring, and respectful

Format your response like this:
🔍 Possible Cause (ممکنہ وجہ):
⚠️ Warning Signs (خطرے کی علامات):
✅ Home Care Tips (گھر میں دیکھ بھال):
🏥 See Doctor If (ڈاکٹر کے پاس جائیں اگر):
"""

def ask_medical_ai(symptoms_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Patient symptoms (علامات): {symptoms_text}"}
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )

    input_len = inputs["input_ids"].shape[1]
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()

# Patient records
patient_records = []

def save_patient_record(name, age, gender, symptoms, ai_advice):
    record = {
        "id": len(patient_records) + 1,
        "name": name, "age": age,
        "gender": gender, "symptoms": symptoms,
        "ai_advice": ai_advice,
    }
    patient_records.append(record)
    return f"✅ Record saved! (Total: {len(patient_records)})"

def get_all_records():
    if not patient_records:
        return "No records yet."
    result = "📋 All Patient Records:\n\n"
    for r in patient_records:
        result += f"---\n🆔 ID: {r['id']}\n👤 {r['name']} | Age: {r['age']} | {r['gender']}\n"
        result += f"🤒 Symptoms: {r['symptoms']}\n💊 Advice: {r['ai_advice'][:200]}...\n\n"
    return result

# ---------- TAB FUNCTIONS ----------
def patient_tab(symptoms):
    if not symptoms.strip():
        return "⚠️ Please enter your symptoms first."
    return ask_medical_ai(symptoms)

def health_worker_tab(name, age, gender, symptoms):
    if not name or not symptoms:
        return "⚠️ Please fill in patient name and symptoms."
    advice = ask_medical_ai(symptoms)
    save_msg = save_patient_record(name, age, gender, symptoms, advice)
    return f"{save_msg}\n\n--- AI Advice ---\n\n{advice}"

def records_tab():
    return get_all_records()

# ---------- BUILD APP ----------
with gr.Blocks(title="🏥 Sehat Sahara", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🏥 Sehat Sahara — صحت سہارا
    ### AI Health Assistant for Rural Pakistan | دیہی پاکستان کے لیے طبی معاون
    *Powered by Gemma 4 | Offline-Ready | Private & Secure*
    ---
    """)

    with gr.Tabs():

        # TAB 1
        with gr.Tab("🤒 Check Symptoms | علامات چیک کریں"):
            gr.Markdown("### Describe your symptoms in Urdu or English")
            symptom_input = gr.Textbox(
                label="Your Symptoms (علامات)",
                placeholder="مثال: مجھے بخار ہے اور سر درد ہے\nExample: I have fever and headache",
                lines=4
            )
            check_btn = gr.Button("🔍 Get Advice | مشورہ لیں", variant="primary")
            advice_output = gr.Textbox(label="AI Medical Advice", lines=15, interactive=False)
            check_btn.click(fn=patient_tab, inputs=[symptom_input], outputs=[advice_output])
            gr.Markdown("⚠️ **This app gives general guidance only. Always consult a real doctor.**")

        # TAB 2
        with gr.Tab("👩‍⚕️ Health Worker | صحت کارکن"):
            gr.Markdown("### Log Patient Information | مریض کی معلومات")
            with gr.Row():
                patient_name = gr.Textbox(label="Patient Name (نام)", placeholder="Ali Khan")
                patient_age  = gr.Number(label="Age (عمر)", value=25)
            patient_gender   = gr.Radio(
                label="Gender (جنس)",
                choices=["Male (مرد)", "Female (عورت)", "Other"],
                value="Male (مرد)"
            )
            patient_symptoms = gr.Textbox(label="Symptoms (علامات)", lines=3)
            save_btn    = gr.Button("💾 Save & Get Advice", variant="primary")
            save_output = gr.Textbox(label="Result", lines=15, interactive=False)
            save_btn.click(
                fn=health_worker_tab,
                inputs=[patient_name, patient_age, patient_gender, patient_symptoms],
                outputs=[save_output]
            )

        # TAB 3
        with gr.Tab("📋 Patient Records | ریکارڈ"):
            view_btn      = gr.Button("🔄 Show All Records", variant="secondary")
            records_output = gr.Textbox(label="Records", lines=20, interactive=False)
            view_btn.click(fn=records_tab, inputs=[], outputs=[records_output])

# Launch
print("🚀 Launching Sehat Sahara...")
app.launch(share=True)

/tmp/ipykernel_57/2670223479.py:85: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="🏥 Sehat Sahara", theme=gr.themes.Soft()) as app:


🚀 Launching Sehat Sahara...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://cabf2da0e023c66f2d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
